# INITIAL IMPORT

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

In [ ]:
from src.config import Configuration
from src.tetris import TetrisConfiguration

T_CONFIG = TetrisConfiguration(
    board_w=10,
    board_h=20,
)

CONFIG = Configuration(
    max_board_size_w=10,
    max_board_size_h=20,
)

# Game

In [ ]:
from src.tetris import Board, PieceEnum, Queue, ActionEnum, ActivePiece, Tetris, RotationEnum, ROTATION_DIR

In [ ]:
game = Tetris()
print('=== SPAWNED from queue ===')
game.print_state()

print('=== LEFT ===')
game.move_active_piece(ActionEnum.LEFT)
game.print_state()

print('=== RIGHT x2 ===')
game.move_active_piece(ActionEnum.RIGHT)
game.move_active_piece(ActionEnum.RIGHT)
game.print_state()

print('=== ROTATE CW ===')
game.move_active_piece(ActionEnum.ROTATE_CW)
game.print_state()

print('=== ROTATE 180 ===')
game.move_active_piece(ActionEnum.ROTATE_180)
game.print_state()

print('=== ROTATE CCW ===')
game.move_active_piece(ActionEnum.ROTATE_CCW)
game.print_state()

print('=== HOLD (swapped with hold slot) ===')
game.move_active_piece(ActionEnum.HOLD)
game.print_state()
print(f'piece={game.active_piece.type} can_hold={game.can_hold}\n')

print('=== DROP + LOCK + SPAWN next ===')
game.move_active_piece(ActionEnum.DROP)
game.print_state()
print(f'piece={game.active_piece.type} pos=({game.active_piece.x},{game.active_piece.y})\n')

print('=== HOLD again (swap back) ===')
lines =game.move_active_piece(ActionEnum.HOLD)
game.print_state()
print(f'piece={game.active_piece.type}\n')

print(f'=== DIRECT: hard_drop + lock_piece (cleared {lines} lines) ===')
lines = game.move_active_piece(ActionEnum.DROP)
game.print_state()
print()

print('=== BOARD: get_ghost_y + hard_drop ===')
gy = game.board.get_ghost_y(game.active_piece)
print(f'ghost Y from y={game.active_piece.y}: {gy}')
drop_dist = game.board.hard_drop(game.active_piece)
print(f'dropped {drop_dist} rows, now at y={game.active_piece.y}')
game.print_state()

# Move Searcher

In [ ]:
from src.tetris import MoveSearcher

game = Tetris(
    # playfield='G'
    playfield=''.join([
        'GGNGGGGGGG',
        'GGNGGGGGGG',
        'GNNNGGGGGG',
        'GNNGGGGGGG',
        'GGNGGGGGGG',
        'NNNGGGGGGG',
        'NNGGGGGGGG',
    ]),
    active_piece='T'
)

game.print_state(include_vanish_zone=True)

all_placements = MoveSearcher(game).get_all_placements()
print(f'Found {len(all_placements)} unique placements for piece')
all_placements

In [ ]:
i = 0

In [ ]:
game.board.print_board(
    b_board=all_placements[i]['bitmap'])
print(f"Placement {i}: {all_placements[i]['state']} lines_cleared={all_placements[i]['lines_cleared']}")
i+=1

### See sequence

In [ ]:
i = 37

In [ ]:
i+=1

In [ ]:
j = 0
game_aux = Tetris(playfield=''.join([
        'GGNGGGGGGG',
        'GGNGGGGGGG',
        'GNNNGGGGGG',
        'GNNGGGGGGG',
        'GGNGGGGGGG',
        'NNNGGGGGGG',
        'NNGGGGGGGG',
    ]),
    active_piece='T')
seq = all_placements[i]['sequence']
print(f'Action sequence to achieve placement {i}: {seq}')

In [ ]:

action = seq[j]
print(f'Action: {action}')
game_aux.move_active_piece(action)
game_aux.print_state(include_vanish_zone=True)
j+=1

# With env

In [ ]:
from src.models import TetrisEnv

env = TetrisEnv(CONFIG, T_CONFIG)

state = env.reset()[0]
print(state.keys())
state

In [ ]:
i = -1

In [ ]:
i+=1
state['boards'][i]


In [ ]:
import numpy as np

grid = state['boards'][i]
# Convert float32 occupancy grid to bitmap (col 0 = LSB)
row_ints = np.zeros(grid.shape[0], dtype=np.uint32)
for x in range(grid.shape[1]):
    row_ints |= (grid[:, x] > 0.5).astype(np.uint32) << x

game.board.color_map = False
game.board.print_board(
    b_board=row_ints,
    include_vanish_zone=True
)
i+=1

In [ ]:
queues = state['queues']
print(queues.shape)
print(queues) 

print()
queues_idx = state['queue_idx']
print(queues_idx.shape)
print(queues_idx) 

In [ ]:
masks = state['placement_mask']
print(masks.shape)
print(masks) 


# Heuristics

In [ ]:
game = Tetris(
    # playfield='G',
    # playfield=''.join([
    #     'GGNGGGGGGG',
    #     'GGNGGGGGGG',
    #     'GNNNGGGGGG',
    #     'GNNGGGGGGG',
    #     'GGNGGGGGGG',
    #     'NNNGGGGGGG',
    #     'NNGGGGGGGG',
    # ]),
    playfield=''.join([
        'NGGGGGGGGG',
        'NGGGGGGGGG',
        'NGGGGGGGGG',
        'NGGGGGGGGG',
        'NGGGGGGGGG',
        # 'GNGGGGGGGG',
    ]),

    active_piece='T'
)
game.print_state()

In [ ]:
pieces = [
    9, 9, 7, 8, 9, 7, 8, 
]

w = 0
for i, p in enumerate(pieces, start=1):
    w += p * i
w

In [ ]:
from src.tetris import HeuristicEvaluator

evaluator = HeuristicEvaluator()

eval_result = evaluator.evaluate(game.board)
print(eval_result)
print(eval_result.compute_total())